# Sembolsüz x86-64 fonksiyon adlandırma — sonraki bağlam deneyi

Bu notebook, assembly metnini ve çağrılan fonksiyon bağlamını kullanarak Qwen2.5-Coder üzerinde 4-bit QLoRA eğitimi yapar. Hedef, proje öneki atılmış fonksiyon adıyla kısa açıklamayı JSON olarak üretmektir; ana test metriği ham `gercek_ad` alanına göre ad F1'dır.

Colab'da önce **Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU** seçin.

## 1. Sabit sürümlü bağımlılıklar

In [ ]:
%pip install -q \
  "transformers==4.48.3" \
  "peft==0.14.0" \
  "trl==0.15.2" \
  "bitsandbytes==0.45.2" \
  "accelerate==1.3.0" \
  "datasets==3.2.0"

## 2. Ayarlar ve GPU'ya göre model seçimi

In [ ]:
from pathlib import Path
import json
import os
import random
import re
import shutil
import zipfile

import torch

# "yukleme": veri-sonraki.zip dosyasını Colab'ın sol dosya panelinden /content'e yükleyin.
# "drive": DRIVE_ZIP yolundaki arşivi kullanın.
VERI_KAYNAGI = "yukleme"                 # "yukleme" veya "drive"
YUKLENEN_ZIP = "/content/veri-sonraki.zip"
DRIVE_ZIP = "/content/drive/MyDrive/asm-adlandirma/veri-sonraki.zip"

# Açıkça başka model yazılabilir. Otomatik seçim açıkken belleği 35 GB'tan az
# GPU'larda bunun 3B karşılığı seçilir.
MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
BELLEGE_GORE_MODEL_SEC = False
L4_MODELI = "Qwen/Qwen2.5-Coder-3B-Instruct"

MAX_UZUNLUK = 3072
EPOCH = 1.0
TRAIN_TAVAN = None       # int verilirse train'den tohumlu rastgele alt küme
VALID_TAVAN = 2000       # değerlendirme hızını sınırlamak için
TEST_DOSYASI = "test_sabit.jsonl"
TEST_N = None            # Yalnız test.jsonl için; None: test kümesinin tamamı
URETIM_BATCH = 16
MAX_YENI_TOKEN = 128
ETKIN_BATCH = 16
MIKRO_BATCH = 2
GRAD_CKPT = True
LORA_R = 16
LORA_ALPHA = 32
OGRENME_ORANI = 1e-4
TOHUM = 7

if not torch.cuda.is_available():
    raise RuntimeError("GPU bulunamadı; Colab çalışma zamanını GPU olarak değiştirin.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("Bu notebook bf16 destekli A100 veya L4 bekliyor.")

gpu = torch.cuda.get_device_properties(0)
# Donanım üreticilerinin kullandığı ondalık GB; 24 GB L4 eşiğini doğru yansıtır.
bellek_gb = gpu.total_memory / 1e9

### Büyük model varyantı (Qwen3-8B)

`BUYUK_MODEL = True` seçildiğinde model ve eğitim ayarları GPU belleğine göre hazırlanır. Önerilen `TRAIN_TAVAN` yalnız yazdırılır; otomatik uygulanmaz.

In [ ]:
from importlib.metadata import version as paket_surumu
from packaging.version import Version

BUYUK_MODEL = False

SECILEN_MODEL = MODEL
if BUYUK_MODEL:
    if Version(paket_surumu("transformers")) < Version("4.51"):
        raise RuntimeError(
            "Qwen3-8B için transformers>=4.51 gerekli. "
            "pip install -q transformers==4.51.3 sonra çalışma zamanını yeniden başlatın."
        )

    MODEL = "Qwen/Qwen3-8B"
    SECILEN_MODEL = MODEL
    MAX_UZUNLUK = 3072
    ETKIN_BATCH = 16
    LORA_R = 16
    LORA_ALPHA = 32
    OGRENME_ORANI = 1e-4

    if bellek_gb < 24:
        raise RuntimeError(
            f"Qwen3-8B varyantı en az 24 GB GPU belleği ister; bulunan: {bellek_gb:.1f} GB."
        )
    if bellek_gb >= 35:
        # Kaba tahmin: A100 40/80 GB'da 4-bit QLoRA ~20-28 GB;
        # 131 bin train satırında 1 epoch yaklaşık 8-12 saat.
        MIKRO_BATCH = 4
        GRAD_CKPT = False
        print("Qwen3-8B / A100 ayarları: 4-bit QLoRA, mikro batch=4, etkin batch=16.")
        print("Kaba tahmin: ~20-28 GB bellek; 131 bin satırda 1 epoch ~8-12 saat.")
        print("Öneri: önce TRAIN_TAVAN=60000 deneyin (otomatik uygulanmadı).")
    else:
        # Kaba tahmin: L4 24 GB'da 4-bit QLoRA ~18-22 GB;
        # 131 bin train satırında 1 epoch yaklaşık 25-35 saat.
        MIKRO_BATCH = 1
        GRAD_CKPT = True
        print("UYARI: Qwen3-8B / L4 ayarları gradient checkpointing ile çok yavaş olabilir.")
        print("Kaba tahmin: ~18-22 GB bellek; 131 bin satırda 1 epoch ~25-35 saat.")
        print("Öneri: önce TRAIN_TAVAN=25000 deneyin (otomatik uygulanmadı).")
    print("Referans: molab 96 GB kartta bf16 LoRA r=16 koşusu yaklaşık 5,5 saat sürdü.")
elif BELLEGE_GORE_MODEL_SEC and bellek_gb < 35:
    SECILEN_MODEL = L4_MODELI

if TEST_DOSYASI not in {"test_sabit.jsonl", "test.jsonl"}:
    raise ValueError("TEST_DOSYASI yalnız 'test_sabit.jsonl' veya 'test.jsonl' olabilir.")
for ad, deger in (
    ("TRAIN_TAVAN", TRAIN_TAVAN),
    ("VALID_TAVAN", VALID_TAVAN),
    ("TEST_N", TEST_N),
):
    if deger is not None and (not isinstance(deger, int) or deger <= 0):
        raise ValueError(f"{ad}, None veya pozitif int olmalı.")
if URETIM_BATCH <= 0 or MAX_YENI_TOKEN <= 0:
    raise ValueError("URETIM_BATCH ve MAX_YENI_TOKEN pozitif olmalı.")
if ETKIN_BATCH % MIKRO_BATCH:
    raise ValueError("ETKIN_BATCH, MIKRO_BATCH değerine tam bölünmeli.")
BIRIKIM_ADIMI = ETKIN_BATCH // MIKRO_BATCH
SABLON_EK = {"enable_thinking": False} if "Qwen3" in SECILEN_MODEL else {}

print(f"GPU: {gpu.name} ({bellek_gb:.1f} GB)")
print(f"Model: {SECILEN_MODEL}")
print(f"Etkin batch: {MIKRO_BATCH} × {BIRIKIM_ADIMI} = {ETKIN_BATCH}")

## 3. Veriyi yükleme veya Drive'dan okuma

JSONL satırları doğrudan okunur; eğitim ve doğrulama `Dataset` nesnelerine yalnız `messages` alanı aktarılır.

In [ ]:
from datasets import Dataset, DatasetDict

if VERI_KAYNAGI == "drive":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    zip_yolu = Path(DRIVE_ZIP)
elif VERI_KAYNAGI == "yukleme":
    zip_yolu = Path(YUKLENEN_ZIP)
else:
    raise ValueError("VERI_KAYNAGI yalnız 'yukleme' veya 'drive' olabilir.")

if not zip_yolu.is_file():
    raise FileNotFoundError(f"Arşiv bulunamadı: {zip_yolu}")

calisma_dizini = Path("/content/asm-adlandirma-sonraki")
acma_dizini = calisma_dizini / "veri-acilmis"
acma_dizini.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_yolu) as z:
    z.extractall(acma_dizini)

# zip_sonraki.sh dosyaları arşiv köküne koyar; iki yaygın alternatif de kabul edilir.
adaylar = [
    acma_dizini,
    acma_dizini / "veri-sonraki",
    acma_dizini / "lora" / "veri-sonraki",
]
gerekli_veri_dosyalari = (
    "train.jsonl", "valid.jsonl", "test.jsonl", "test_sabit.jsonl"
)
VERI_DIZINI = next(
    (p for p in adaylar if all((p / dosya).is_file() for dosya in gerekli_veri_dosyalari)),
    None,
)
if VERI_DIZINI is None:
    raise FileNotFoundError(
        "Arşivde train.jsonl, valid.jsonl, test.jsonl ve test_sabit.jsonl birlikte bulunamadı."
    )

def jsonl_oku(yol):
    satirlar = []
    with yol.open(encoding="utf-8") as f:
        for satir_no, satir in enumerate(f, 1):
            if not satir.strip():
                continue
            try:
                satirlar.append(json.loads(satir))
            except json.JSONDecodeError as hata:
                raise ValueError(f"Geçersiz JSON: {yol}:{satir_no}") from hata
    return satirlar

def tohumlu_alt_kume(satirlar, tavan, tohum):
    if tavan is None or len(satirlar) <= tavan:
        return list(satirlar)
    return random.Random(tohum).sample(satirlar, tavan)

train_ham = jsonl_oku(VERI_DIZINI / "train.jsonl")
valid_ham = jsonl_oku(VERI_DIZINI / "valid.jsonl")
train_secili = tohumlu_alt_kume(train_ham, TRAIN_TAVAN, TOHUM)
valid_secili = tohumlu_alt_kume(valid_ham, VALID_TAVAN, TOHUM)

# Fazladan id/proje/opt/baglam_var/gercek_ad alanlarını Dataset'e taşımıyoruz.
ham_veri = DatasetDict({
    "train": Dataset.from_list([{"messages": r["messages"]} for r in train_secili]),
    "validation": Dataset.from_list([{"messages": r["messages"]} for r in valid_secili]),
})

print(f"train: {len(train_ham):>6} satır → {len(train_secili):>6} seçildi")
print(f"valid: {len(valid_ham):>6} satır → {len(valid_secili):>6} seçildi")
with (VERI_DIZINI / TEST_DOSYASI).open(encoding="utf-8") as f:
    test_adedi = sum(1 for satir in f if satir.strip())
print(f"{TEST_DOSYASI:>16}: {test_adedi:>6} satır")

## 4. 4-bit model ve tüm lineer katmanlarda LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

set_seed(TOHUM)
tokenizer = AutoTokenizer.from_pretrained(SECILEN_MODEL, use_fast=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

niceleme = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    SECILEN_MODEL,
    quantization_config=niceleme,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    attn_implementation="sdpa",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=GRAD_CKPT,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
model = get_peft_model(
    model,
    LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules="all-linear",
    ),
)
model.print_trainable_parameters()

## 5. Sohbet şablonu ve yalnız assistant tokenlarında kayıp

Qwen sohbet şablonuyla kullanıcı istemi ve tam konuşma ayrı kodlanır. Tam konuşma `MAX_UZUNLUK` değerini aşarsa assistant yanıtını kesmek yerine örneğin tamamı atılır.

In [ ]:
def kodla(ornek):
    mesajlar = ornek["messages"]
    if [m["role"] for m in mesajlar] != ["system", "user", "assistant"]:
        raise ValueError("Beklenen mesaj sırası: system, user, assistant")

    istem = tokenizer.apply_chat_template(
        mesajlar[:2], tokenize=True, add_generation_prompt=True, **SABLON_EK
    )
    tumu = tokenizer.apply_chat_template(
        mesajlar, tokenize=True, add_generation_prompt=False, **SABLON_EK
    )
    if tumu[:len(istem)] != istem:
        raise ValueError("Sohbet şablonunda assistant başlangıcı belirlenemedi.")

    gecerli = len(tumu) <= MAX_UZUNLUK and len(tumu) > len(istem)
    if not gecerli:
        return {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
            "uzunluk": len(tumu),
            "gecerli": False,
        }

    etiketler = [-100] * len(istem) + tumu[len(istem):]
    return {
        "input_ids": tumu,
        "attention_mask": [1] * len(tumu),
        "labels": etiketler,
        "uzunluk": len(tumu),
        "gecerli": True,
    }

tokenli_veri = ham_veri.map(
    kodla,
    remove_columns=ham_veri["train"].column_names,
    desc="Qwen sohbet şablonu uygulanıyor",
)

for bolum in ("train", "validation"):
    once = len(tokenli_veri[bolum])
    tokenli_veri[bolum] = tokenli_veri[bolum].filter(
        lambda ornek: ornek["gecerli"],
        desc=f"{bolum}: uzun örnekler atılıyor",
    )
    atilan = once - len(tokenli_veri[bolum])
    if not tokenli_veri[bolum]:
        raise ValueError(f"{bolum} bölümünde geçerli örnek kalmadı.")
    uzunluklar = tokenli_veri[bolum]["uzunluk"]
    print(
        f"{bolum:>10}: {len(uzunluklar)} örnek; {atilan} uzun örnek atıldı; "
        f"ortalama {sum(uzunluklar) / len(uzunluklar):.0f}, en uzun {max(uzunluklar)} token"
    )

tokenli_veri = tokenli_veri.remove_columns(["uzunluk", "gecerli"])

## 6. Eğitim öncesi duman denetimi

İlk kodlanmış örnekte istem etiketlerinin tamamının `-100`, kayba giren kısmın ise yalnız assistant yanıtı olduğu doğrulanıp gösterilir.

In [ ]:
ilk_ornek = tokenli_veri["train"][0]
ilk_aktif = next(i for i, etiket in enumerate(ilk_ornek["labels"]) if etiket != -100)
assert all(etiket == -100 for etiket in ilk_ornek["labels"][:ilk_aktif])
assert all(etiket != -100 for etiket in ilk_ornek["labels"][ilk_aktif:])

assistant_tokenlari = [
    etiket for etiket in ilk_ornek["labels"] if etiket != -100
]
print(f"Maskeli istem tokenı: {ilk_aktif}")
print(f"Assistant tokenı: {len(assistant_tokenlari)}")
print("Kayba giren assistant bölümü:")
print(tokenizer.decode(assistant_tokenlari, skip_special_tokens=False))

## 7. Eğitim

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

egitim_ayari = TrainingArguments(
    output_dir="/content/asm-adlandirma-sonraki/denetim-noktalari",
    num_train_epochs=EPOCH,
    per_device_train_batch_size=MIKRO_BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=BIRIKIM_ADIMI,
    learning_rate=OGRENME_ORANI,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_8bit",
    bf16=True,
    tf32=True,
    gradient_checkpointing=GRAD_CKPT,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=0.3,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    logging_steps=10,
    report_to="none",
    seed=TOHUM,
    data_seed=TOHUM,
    group_by_length=True,
    remove_unused_columns=False,
)

veri_toplayici = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
    return_tensors="pt",
)
egitici = Trainer(
    model=model,
    args=egitim_ayari,
    train_dataset=tokenli_veri["train"],
    eval_dataset=tokenli_veri["validation"],
    data_collator=veri_toplayici,
)
egitim_sonucu = egitici.train()
egitici.evaluate()
egitim_sonucu

## 8. Adaptörü Drive'a kaydetme ve zip olarak indirme

In [ ]:
from google.colab import drive, files

drive.mount("/content/drive", force_remount=False)
model_kisa = SECILEN_MODEL.rsplit("/", 1)[-1]
ADAPTOR_DIZINI = Path("/content/drive/MyDrive/asm-adlandirma") / f"adaptor-sonraki-{model_kisa}"
ADAPTOR_DIZINI.parent.mkdir(parents=True, exist_ok=True)

egitici.save_model(str(ADAPTOR_DIZINI))
tokenizer.save_pretrained(str(ADAPTOR_DIZINI))

zip_koku = Path("/content") / ADAPTOR_DIZINI.name
zip_yolu = Path(shutil.make_archive(
    str(zip_koku),
    "zip",
    root_dir=ADAPTOR_DIZINI.parent,
    base_dir=ADAPTOR_DIZINI.name,
))
print(f"Drive: {ADAPTOR_DIZINI}")
print(f"İndirilecek arşiv: {zip_yolu} ({zip_yolu.stat().st_size / 1024**2:.0f} MB)")
files.download(str(zip_yolu))

## 9. Test ölçümü

Varsayılan sabit test kümesi dosya sırası korunarak değerlendirilir. `test.jsonl` seçilirse tohumlu karıştırma ve `TEST_N` sınırı uygulanır. Üretim sol dolgulu batch'lerde greedy yapılır. Ana `f1` ham `gercek_ad`, `f1_oneksiz` ise assistant hedefindeki öneksiz ad üzerinden hesaplanır.

In [ ]:
import gc
from tqdm.auto import tqdm

# taban.py içindeki bu iki fonksiyon birebir kopyalanmıştır.
def kelimeler(ad: str) -> list[str]:
    ad = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", ad)
    return [k for k in re.split(r"[_\W]+", ad.lower()) if k]

def f1(tahmin: str, gercek: str) -> float:
    t, g = kelimeler(tahmin), kelimeler(gercek)
    ortak = len(set(t) & set(g))
    if not ortak:
        return 0.0
    p, r = ortak / len(set(t)), ortak / len(set(g))
    return 2 * p * r / (p + r)

del egitici
gc.collect()
torch.cuda.empty_cache()
model.config.use_cache = True
model.eval()
tokenizer.padding_side = "left"

test_satirlari = jsonl_oku(VERI_DIZINI / TEST_DOSYASI)
if TEST_DOSYASI == "test.jsonl":
    random.Random(TOHUM).shuffle(test_satirlari)
    if TEST_N is not None:
        test_satirlari = test_satirlari[:TEST_N]
print(f"Test dosyası: {TEST_DOSYASI}; seçilen örnek: {len(test_satirlari)}")

def yeni_token_sayisi(tokenlar):
    eos = model.generation_config.eos_token_id
    if eos is None:
        eos = tokenizer.eos_token_id
    eos_kumesi = set(eos if isinstance(eos, list) else [eos])
    for sira, token in enumerate(tokenlar.tolist(), 1):
        if token in eos_kumesi:
            return sira
    return len(tokenlar)

sonuclar = []
json_hatasi = 0
for baslangic in tqdm(
    range(0, len(test_satirlari), URETIM_BATCH),
    desc="Batch greedy test üretimi",
):
    batch = test_satirlari[baslangic:baslangic + URETIM_BATCH]
    metinler = [
        tokenizer.apply_chat_template(
            kayit["messages"][:2],
            tokenize=False,
            add_generation_prompt=True,
            **SABLON_EK,
        )
        for kayit in batch
    ]
    girdiler = tokenizer(metinler, padding=True, return_tensors="pt")
    girdi_uzunluklari = girdiler["attention_mask"].sum(dim=1).tolist()
    fazla = [
        kayit.get("id", "")
        for kayit, uzunluk in zip(batch, girdi_uzunluklari)
        if uzunluk > MAX_UZUNLUK
    ]
    if fazla:
        raise ValueError(
            f"Test girdisi {MAX_UZUNLUK} tokenı aşıyor; örnek id: {fazla[:5]}"
        )
    girdiler = {ad: deger.to(model.device) for ad, deger in girdiler.items()}

    with torch.inference_mode():
        uretilen = model.generate(
            **girdiler,
            do_sample=False,
            max_new_tokens=MAX_YENI_TOKEN,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    yeni_tokenlar = uretilen[:, girdiler["input_ids"].shape[1]:]
    yanitlar = tokenizer.batch_decode(yeni_tokenlar, skip_special_tokens=True)

    for kayit, metin, tokenlar, girdi_uzunlugu in zip(
        batch, yanitlar, yeni_tokenlar, girdi_uzunluklari
    ):
        eslesmeler = re.findall(r"\{[^{}]*\}", metin)
        try:
            if not eslesmeler:
                raise ValueError("JSON nesnesi bulunamadı")
            cevap = json.loads(eslesmeler[-1])
            if not isinstance(cevap, dict):
                raise ValueError("Son JSON nesnesi sözlük değil")
        except (json.JSONDecodeError, ValueError):
            json_hatasi += 1
            cevap = {}

        tahmin = str(cevap.get("ad", ""))
        aciklama = str(cevap.get("aciklama", ""))
        ogretmen_cevabi = json.loads(kayit["messages"][2]["content"])
        hedef_ad = str(ogretmen_cevabi["ad"])
        gercek = str(kayit["gercek_ad"])
        skor = f1(tahmin, gercek)
        skor_oneksiz = f1(tahmin, hedef_ad)
        sonuc = {
            "id": kayit.get("id", ""),
            "gercek": gercek,
            "hedef_ad": hedef_ad,
            "tahmin": tahmin,
            "aciklama": aciklama,
            "ogretmen": str(ogretmen_cevabi.get("aciklama", "")),
            "f1": skor,
            "f1_oneksiz": skor_oneksiz,
            "opt": kayit.get("opt", ""),
            # Token tavanı bağlamı tümden atmış olabilir: girdideki başlığa bakılır.
            "baglam_var": "; --- çağrılan fonksiyonlar ---" in kayit["messages"][1]["content"],
            "token": int(girdi_uzunlugu + yeni_token_sayisi(tokenlar)),
        }
        sonuclar.append(sonuc)

sonuc_yolu = Path("/content") / f"sonuc-sonraki-{model_kisa}.jsonl"
with sonuc_yolu.open("w", encoding="utf-8") as f:
    for sonuc in sonuclar:
        f.write(json.dumps(sonuc, ensure_ascii=False) + "\n")

def ozet_yaz(etiket, satirlar):
    if not satirlar:
        print(f"{etiket}: örnek yok")
        return
    ortalama = sum(r["f1"] for r in satirlar) / len(satirlar)
    ortalama_oneksiz = sum(r["f1_oneksiz"] for r in satirlar) / len(satirlar)
    tam = sum(r["f1"] == 1.0 for r in satirlar)
    print(
        f"{etiket}: ortalama f1={ortalama:.3f}, "
        f"ortalama f1_oneksiz={ortalama_oneksiz:.3f}, "
        f"n={len(satirlar)}, tam isabet={tam}"
    )

print("\nÖzet — opt")
for opt in sorted({r["opt"] for r in sonuclar}):
    ozet_yaz(opt, [r for r in sonuclar if r["opt"] == opt])
print("\nÖzet — bağlam")
for baglam_var in (True, False):
    ozet_yaz(
        f"baglam_var={baglam_var}",
        [r for r in sonuclar if r["baglam_var"] is baglam_var],
    )
print("\nÖzet — genel")
ozet_yaz("genel", sonuclar)
print(f"JSON ayrıştırma başarısızlığı: {json_hatasi}")
print(f"ayrıntı → {sonuc_yolu}")
files.download(str(sonuc_yolu))